# Notebook 4b — Ensembles, Evaluation & Final Analysis
## HMDA 2023 Loan Denial Prediction | Fresh JVM Session

---

### What this notebook does

1. **Loads saved models** from Notebook 4a (LR, RF, GBT, DT) — no retraining
2. **Ensemble engineering**: Soft Voting, Weighted Voting, Stacking
3. **Threshold optimization** for best model
4. **Error analysis** and model agreement
5. **Feature importance** cross-model comparison
6. **Final leaderboard** and artifact saving

### Why a separate notebook?

Starting a fresh Spark session gives us a **clean 7 GB JVM heap**. In NB4a, by
the time we reached ensembles, the heap was full of accumulated model objects
and cached DataFrames. Here we only load the 3-4 models we actually need.

In [1]:
# ============================================================
# Cell 1: Imports & Fresh SparkSession
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml.linalg import Vectors, VectorUDT, DenseVector
from pyspark.ml.classification import (
    LogisticRegression, LogisticRegressionModel,
    DecisionTreeClassificationModel,
    RandomForestClassificationModel,
    GBTClassificationModel,
    NaiveBayes,
    GBTClassifier,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.feature import VectorAssembler

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import json, time, os, warnings
warnings.filterwarnings("ignore")

try:
    import mlflow
    import mlflow.spark
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False

# Fresh Spark session — clean JVM
spark = (SparkSession.builder
    .appName("HMDA_2023_Ensembles_4b")
    .master("local[2]")
    .config("spark.driver.memory", "6g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.driver.extraJavaOptions",
            "-XX:+UseG1GC -XX:G1HeapRegionSize=16m "
            "-XX:InitiatingHeapOccupancyPercent=35")
    .config("spark.memory.fraction", "0.6")
    .config("spark.memory.storageFraction", "0.3")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Fresh Spark session: {spark.version} | Cores: {spark.sparkContext.defaultParallelism}")

26/03/01 20:45:15 WARN Utils: Your hostname, Adityas-Laptop.local resolves to a loopback address: 127.0.0.1; using 10.150.186.221 instead (on interface en0)
26/03/01 20:45:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 20:45:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Fresh Spark session: 3.5.8 | Cores: 2


In [2]:
# ============================================================
# Cell 2: Load Data, Models & NB4a Results
# ============================================================

DATA_DIR  = "/Users/adi/Desktop/assignmt/project/data/processed"
MODEL_DIR = f"{DATA_DIR}/models"

# Load feature-engineered data
train_df = spark.read.parquet(f"{DATA_DIR}/train_features.parquet")
test_df  = spark.read.parquet(f"{DATA_DIR}/test_features.parquet")

# Load metadata
with open(f"{DATA_DIR}/feature_metadata.json") as f:
    feature_meta = json.load(f)
feature_dim = feature_meta["n_features"]

# Add class weights
with open(f"{DATA_DIR}/best_params_4a.json") as f:
    BEST_PARAMS = json.load(f)

label_counts = {row["label"]: row["count"]
                for row in train_df.groupBy("label").count().collect()}
n_orig, n_denied = label_counts[0.0], label_counts[1.0]
total = n_orig + n_denied
w0, w1 = total / (2.0 * n_orig), total / (2.0 * n_denied)

train_w = train_df.withColumn(
    "weight", F.when(F.col("label") == 1.0, F.lit(w1)).otherwise(F.lit(w0))
).cache()
test_w = test_df.withColumn(
    "weight", F.when(F.col("label") == 1.0, F.lit(w1)).otherwise(F.lit(w0))
).cache()

train_count = train_w.count()
test_count  = test_w.count()

print(f"Train: {train_count:,} | Test: {test_count:,} | Features: {feature_dim}")
print(f"Class weights: originated={w0:.4f}, denied={w1:.4f}")

# Load NB4a results
with open(f"{DATA_DIR}/model_results_4a.json") as f:
    ALL_RESULTS = json.load(f)
print(f"\nLoaded {len(ALL_RESULTS)} model results from NB4a")

# Load ensemble prediction samples
with open(f"{DATA_DIR}/ensemble_predictions_4a.json") as f:
    preds_raw = json.load(f)
ALL_PREDICTIONS = {k: pd.DataFrame(v) for k, v in preds_raw.items()}
print(f"Loaded ensemble predictions for: {list(ALL_PREDICTIONS.keys())}")

# Load saved model objects
print("\nLoading saved models...")
models = {}
for name, cls in [("LR", LogisticRegressionModel),
                  ("DT", DecisionTreeClassificationModel),
                  ("RF", RandomForestClassificationModel),
                  ("GBT", GBTClassificationModel)]:
    path = f"{MODEL_DIR}/best_{name.lower()}"
    try:
        models[name] = cls.load(path)
        print(f"  Loaded {name} from {path}")
    except Exception as e:
        print(f"  {name} not found: {e}")

print(f"\nModels available: {list(models.keys())}")

# Evaluators
eval_roc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
eval_pr  = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderPR")
eval_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# Extract p1 helper
extract_p1 = F.udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.5, DoubleType())


def evaluate_model(predictions, model_name, train_time=0.0, params=None,
                   tags=None, log_mlflow=True):
    metrics = {}
    try: metrics["ROC-AUC"] = eval_roc.evaluate(predictions)
    except: metrics["ROC-AUC"] = 0.5
    try: metrics["PR-AUC"] = eval_pr.evaluate(predictions)
    except: metrics["PR-AUC"] = 0.0
    metrics["F1"]       = eval_f1.evaluate(predictions)
    metrics["Accuracy"] = eval_acc.evaluate(predictions)
    cm = (predictions
        .select(F.col("label").cast("int").alias("actual"),
                F.col("prediction").cast("int").alias("pred"))
        .groupBy("actual", "pred").count().collect())
    cm_dict = {(r["actual"], r["pred"]): r["count"] for r in cm}
    tp, fp = cm_dict.get((1,1),0), cm_dict.get((0,1),0)
    fn, tn = cm_dict.get((1,0),0), cm_dict.get((0,0),0)
    d_prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    d_rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    d_f1   = 2*d_prec*d_rec/(d_prec+d_rec) if (d_prec+d_rec)>0 else 0.0
    metrics.update({
        "Denial_Precision": d_prec, "Denial_Recall": d_rec, "Denial_F1": d_f1,
        "Train_Time_s": train_time,
        "Confusion": {"TP": tp, "FP": fp, "FN": fn, "TN": tn}
    })
    ALL_RESULTS[model_name] = metrics
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  PR-AUC: {metrics['PR-AUC']:.4f}  |  ROC-AUC: {metrics['ROC-AUC']:.4f}")
    print(f"  Denial P/R/F1: {d_prec:.4f} / {d_rec:.4f} / {d_f1:.4f}")
    print(f"  Confusion:  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
    print(f"  Time: {train_time:.1f}s")
    if log_mlflow and MLFLOW_AVAILABLE:
        mlflow.set_tracking_uri(f"file:///{DATA_DIR}/mlruns")
        mlflow.set_experiment("HMDA_2023_Model_Training")
        with mlflow.start_run(run_name=model_name):
            if params:
                for k, v in params.items(): mlflow.log_param(k, v)
            for k, v in metrics.items():
                if k != "Confusion" and isinstance(v, (int, float)):
                    mlflow.log_metric(k.replace("-", "_"), v)
    return metrics

26/03/01 20:45:29 WARN MemoryStore: Not enough space to cache rdd_15_4 in memory! (computed 452.1 MiB so far)
26/03/01 20:45:29 WARN BlockManager: Persisting block rdd_15_4 to disk instead.
26/03/01 20:45:32 WARN MemoryStore: Not enough space to cache rdd_15_4 in memory! (computed 452.1 MiB so far)
26/03/01 20:45:33 WARN MemoryStore: Not enough space to cache rdd_15_4 in memory! (computed 452.1 MiB so far)


Train: 6,148,713 | Test: 1,533,840 | Features: 145
Class weights: originated=0.6753, denied=1.9264

Loaded 8 model results from NB4a
Loaded ensemble predictions for: ['NaiveBayes', 'LR', 'SVM', 'DT', 'RF', 'GBT', 'MLP']

Loading saved models...
  Loaded LR from /Users/adi/Desktop/assignmt/project/data/processed/models/best_lr
  Loaded DT from /Users/adi/Desktop/assignmt/project/data/processed/models/best_dt


  Loaded RF from /Users/adi/Desktop/assignmt/project/data/processed/models/best_rf
  Loaded GBT from /Users/adi/Desktop/assignmt/project/data/processed/models/best_gbt

Models available: ['LR', 'DT', 'RF', 'GBT']


## Section 3 — Ensemble Engineering: The Science of Model Combination

### Why ensembles work: Condorcet's Jury Theorem

If each "voter" (model) is correct >50% of the time AND errors are **independent**, majority voting is exponentially more reliable as voters increase.

With 3 models at 75% individual accuracy and independent errors:
```
P(majority correct) = C(3,2)*0.75²*0.25 + C(3,3)*0.75³ = 0.844
```

84.4% > 75% — a significant lift from mere voting.

**The catch**: If models make the SAME errors (correlated), combining doesn't help.

---

### How to Select Models for an Ensemble (The Complete Framework)

This is the **key interview question**: *"How do you decide which models to put in an ensemble?"*

#### Step 1: Measure Prediction Diversity

Two models that always agree are redundant. We need models that **disagree on different samples**.

**Metrics for diversity**:

| Metric | Formula | Interpretation |
|:-------|:--------|:--------------|
| **Error correlation** | corr(errors_A, errors_B) | Low = diverse (good) |
| **Disagreement rate** | % of samples where A and B predict differently | High = diverse (good) |
| **Q-statistic** | (N11*N00 - N01*N10) / (N11*N00 + N01*N10) | -1 to 1; low = diverse |
| **Double-fault rate** | % where both A and B are wrong | Low = complementary (good) |

#### Step 2: Balance Accuracy vs Diversity

A random model is perfectly diverse but useless. The sweet spot:
- Each model individually performant (PR-AUC > some threshold)
- Low error correlation between selected models

**Practical rule**: Pick the top-K models by individual performance, then filter for diversity.

#### Step 3: Select Ensemble Architecture

| Architecture | How it combines | Best when |
|:------------|:---------------|:----------|
| **Hard voting** | Majority class | Models are similarly accurate |
| **Soft voting** | Average P(class) | Models have good probability calibration |
| **Weighted voting** | Weighted average | Models differ significantly in quality |
| **Stacking (blending)** | Meta-learner on predictions | Models capture different patterns |
| **Multi-level stacking** | Stack of stacks | Maximum performance, complexity budget allows |

#### Step 4: Validate Ensemble Benefit

The ensemble must beat the best individual model on the **test set** (not just validation). If it doesn't, the simpler individual model wins.

---

### Our Ensemble Strategy

We'll build 3 ensemble variants (Two-Tier Stacking dropped per OOM analysis — typical lift < 0.5% PR-AUC, not worth the memory cost):

| Ensemble | Base models | Architecture | Rationale |
|:---------|:-----------|:-------------|:----------|
| **A: All-Vote** | All models | Soft voting | Maximum diversity |
| **B: Top-3 Diverse** | Selected by error correlation | Weighted voting | Optimal accuracy-diversity trade-off |
| **C: Stacking** | Top-3 | LR meta-learner (blending) | Learn optimal non-uniform combination |
| **D: Two-Tier Stack** | All → LR → GBT meta | Multi-level | Maximum performance extraction |

In [3]:
# ============================================================
# Cell 16: Ensemble Model Selection — Diversity Analysis
# ============================================================

print("=" * 70)
print("ENSEMBLE MODEL SELECTION: DIVERSITY ANALYSIS")
print("=" * 70)

# ALL_PREDICTIONS now stores pandas DataFrames (sampled 5%)
# No Spark joins needed — pure pandas analysis (fast + memory-safe)

available = {k: v for k, v in ALL_PREDICTIONS.items()}
print(f"Available models for ensemble: {list(available.keys())}")

# Combine all predictions into one pandas DF
import pandas as pd

base_pdf = None
pred_cols = []

for mname, pdf in available.items():
    col = f"pred_{mname}"
    pred_cols.append(col)
    
    if base_pdf is None:
        base_pdf = pdf.rename(columns={"prob": col}).reset_index(drop=True)
    else:
        # Align by index (all sampled with same seed, same size)
        min_len = min(len(base_pdf), len(pdf))
        base_pdf = base_pdf.iloc[:min_len].copy()
        base_pdf[col] = pdf["prob"].values[:min_len]

print(f"\nSample size for diversity analysis: {len(base_pdf):,}")

# Compute error correlation matrix
y_true = base_pdf["label"].values

error_vectors = {}
for col in pred_cols:
    preds = (base_pdf[col].values > 0.5).astype(int)
    errors = (preds != y_true).astype(int)
    error_vectors[col.replace("pred_", "")] = errors

error_df = pd.DataFrame(error_vectors)
corr_matrix = error_df.corr()

print("\n" + "=" * 50)
print("ERROR CORRELATION MATRIX")
print("=" * 50)
print("(Low correlation = diverse errors = good for ensembles)")
print(corr_matrix.round(3).to_string())

# Disagreement rates
print("\n" + "=" * 50)
print("PAIRWISE DISAGREEMENT RATES")
print("=" * 50)
print("(High = models disagree = good for ensembles)")

model_names_list = list(error_vectors.keys())
disagree_matrix = pd.DataFrame(index=model_names_list, columns=model_names_list, dtype=float)

for i, m1 in enumerate(model_names_list):
    for j, m2 in enumerate(model_names_list):
        if i == j:
            disagree_matrix.loc[m1, m2] = 0.0
        else:
            disagree = (error_vectors[m1] != error_vectors[m2]).mean()
            disagree_matrix.loc[m1, m2] = round(disagree, 3)

print(disagree_matrix.to_string())

# Recommend ensemble composition
individual_prauc = {}
for mname in available.keys():
    matching = [k for k in ALL_RESULTS if mname.lower() in k.lower().replace("_", "")]
    if matching:
        individual_prauc[mname] = ALL_RESULTS[matching[0]]["PR-AUC"]

print("\n" + "=" * 50)
print("INDIVIDUAL PR-AUC:")
for m, s in sorted(individual_prauc.items(), key=lambda x: -x[1]):
    print(f"  {m:<20} {s:.4f}")

# Select top-3 diverse
sorted_models = sorted(individual_prauc.items(), key=lambda x: -x[1])
top3 = [m for m, _ in sorted_models[:3]]
print(f"\nSelected for ensemble: {top3}")
print(f"  Mean pairwise error correlation among selected:")
for i in range(len(top3)):
    for j in range(i+1, len(top3)):
        if top3[i] in corr_matrix.columns and top3[j] in corr_matrix.columns:
            print(f"    {top3[i]} vs {top3[j]}: {corr_matrix.loc[top3[i], top3[j]]:.3f}")

ENSEMBLE MODEL SELECTION: DIVERSITY ANALYSIS
Available models for ensemble: ['NaiveBayes', 'LR', 'SVM', 'DT', 'RF', 'GBT', 'MLP']

Sample size for diversity analysis: 76,384

ERROR CORRELATION MATRIX
(Low correlation = diverse errors = good for ensembles)
            NaiveBayes     LR    SVM     DT     RF    GBT    MLP
NaiveBayes       1.000  0.661 -0.767  0.485  0.674  0.524  0.753
LR               0.661  1.000 -0.801  0.612  0.790  0.687  0.716
SVM             -0.767 -0.801  1.000 -0.592 -0.837 -0.626 -0.894
DT               0.485  0.612 -0.592  1.000  0.694  0.766  0.545
RF               0.674  0.790 -0.837  0.694  1.000  0.748  0.771
GBT              0.524  0.687 -0.626  0.766  0.748  1.000  0.580
MLP              0.753  0.716 -0.894  0.545  0.771  0.580  1.000

PAIRWISE DISAGREEMENT RATES
(High = models disagree = good for ensembles)
            NaiveBayes     LR    SVM     DT     RF    GBT    MLP
NaiveBayes       0.000  0.005  0.997  0.007  0.005  0.007  0.004
LR               0.

In [4]:
# ============================================================
# Cell 17: Ensemble A — Soft Voting (All Models)
# ============================================================

print("=" * 70)
print("ENSEMBLE A: SOFT VOTING (ALL MODELS)")
print("=" * 70)

start = time.time()

print("Generating predictions from loaded models...")
test_idx = test_w.withColumn("_idx", F.monotonically_increasing_id())
vote_probs = test_idx.select("_idx", "label")
prob_cols = []

for mname, model in models.items():
    col = f"p_{mname}"
    prob_cols.append(col)
    preds = model.transform(test_idx).select(
        "_idx", extract_p1("probability").alias(col))
    vote_probs = vote_probs.join(preds, "_idx")

avg_expr = sum(F.col(c) for c in prob_cols) / float(len(prob_cols))
make_vec = F.udf(lambda p: Vectors.dense([1.0-p, p]), VectorUDT())

vote_a_preds = vote_probs.select(
    "label",
    F.when(avg_expr > 0.5, 1.0).otherwise(0.0).alias("prediction"),
    make_vec(avg_expr).alias("rawPrediction"),
    make_vec(avg_expr).alias("probability"),
)

evaluate_model(vote_a_preds, "E_A_SoftVote_All", time.time()-start,
    params={"n_models": len(prob_cols), "method": "average"},
    tags={"family": "ensemble", "type": "voting"})

vote_probs.cache()
vote_probs.count()
print(f"  Ensemble A done. {len(prob_cols)} models averaged.")

ENSEMBLE A: SOFT VOTING (ALL MODELS)
Generating predictions from loaded models...


26/03/01 20:45:43 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/03/01 20:46:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/03/01 20:46:58 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/03/01 20:47:09 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:47:23 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/03/01 20:48:21 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:48:34 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/03/01 20:49:30 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:49:33 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/03/01 20:50:24 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:50:28 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/03/01 


  E_A_SoftVote_All
  PR-AUC: 0.9988  |  ROC-AUC: 0.9996
  Denial P/R/F1: 0.9887 / 0.9895 / 0.9891
  Confusion:  TN=1,132,284  FP=4,493  FN=4,176  TP=392,887
  Time: 0.3s


26/03/01 20:51:20 WARN DAGScheduler: Broadcasting large task binary with size 6.2 MiB
26/03/01 20:52:11 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:52:19 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB


  Ensemble A done. 4 models averaged.


In [5]:
# ============================================================
# Cell 18: Ensemble B — Weighted Voting (Top-3 by PR-AUC)
# ============================================================

print("=" * 70)
print("ENSEMBLE B: WEIGHTED VOTING (TOP-3)")
print("=" * 70)

start = time.time()

# Get individual PR-AUC from loaded results
individual_prauc = {}
for mname in ALL_PREDICTIONS.keys():
    matching = [k for k in ALL_RESULTS if mname.lower() in k.lower().replace("_", "")]
    if matching:
        individual_prauc[mname] = ALL_RESULTS[matching[0]]["PR-AUC"]

sorted_models = sorted(individual_prauc.items(), key=lambda x: -x[1])
top3 = [m for m, _ in sorted_models[:3]]
print(f"Top-3 by PR-AUC: {top3}")

weights = {m: individual_prauc.get(m, 0.5) for m in top3}
total_wt = sum(weights.values())
norm_weights = {m: w/total_wt for m, w in weights.items()}

for m, w in norm_weights.items():
    print(f"  {m}: {w:.4f} (raw PR-AUC: {weights[m]:.4f})")

name_to_col = {"NB": "p_NB", "LR": "p_LR", "DT": "p_DT",
               "RF": "p_RF", "GBT": "p_GBT",
               "NaiveBayes": "p_NB", "LogisticRegression": "p_LR",
               "DecisionTree": "p_DT", "RandomForest": "p_RF"}

top3_cols, top3_weights = [], []
for m in top3:
    col = name_to_col.get(m, f"p_{m}")
    if col in vote_probs.columns:
        top3_cols.append(col)
        top3_weights.append(norm_weights[m])

if not top3_cols:
    top3_cols = prob_cols
    top3_weights = [1.0/len(prob_cols)] * len(prob_cols)

weighted_expr = sum(F.col(c) * w for c, w in zip(top3_cols, top3_weights))
make_vec = F.udf(lambda p: Vectors.dense([1.0-p, p]), VectorUDT())

vote_b_preds = vote_probs.select(
    "label",
    F.when(weighted_expr > 0.5, 1.0).otherwise(0.0).alias("prediction"),
    make_vec(weighted_expr).alias("rawPrediction"),
    make_vec(weighted_expr).alias("probability"),
)

evaluate_model(vote_b_preds, "E_B_WeightedVote_Top3", time.time()-start,
    params={"models": str(top3), "weights": str(dict(zip(top3_cols, top3_weights)))},
    tags={"family": "ensemble", "type": "weighted_voting"})

vote_probs.unpersist()

ENSEMBLE B: WEIGHTED VOTING (TOP-3)
Top-3 by PR-AUC: ['GBT', 'SVM', 'NaiveBayes']
  GBT: 0.3341 (raw PR-AUC: 0.9989)
  SVM: 0.3336 (raw PR-AUC: 0.9976)
  NaiveBayes: 0.3323 (raw PR-AUC: 0.9937)


26/03/01 20:52:25 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:52:55 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:53:23 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:53:38 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 20:53:54 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB



  E_B_WeightedVote_Top3
  PR-AUC: 0.9990  |  ROC-AUC: 0.9997
  Denial P/R/F1: 0.0000 / 0.0000 / 0.0000
  Confusion:  TN=1,136,777  FP=0  FN=397,063  TP=0
  Time: 0.0s


26/03/01 20:54:00 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB


DataFrame[_idx: bigint, label: double, p_LR: double, p_DT: double, p_RF: double, p_GBT: double]

In [6]:
# ============================================================
# Cell 19: Ensemble C — Stacking (Blending) with Meta-Learner
# ============================================================
# OOM FIX: Uses loaded models (no re-training). Only trains
# a tiny meta-learner on blend_val predictions.

print("=" * 70)
print("ENSEMBLE C: STACKING (BLENDING) — LR META-LEARNER")
print("=" * 70)

start = time.time()

# Split train for blending
splits = train_w.randomSplit([0.7, 0.3], seed=42)
blend_val = splits[1].cache()
blend_count = blend_val.count()
print(f"Blend validation set: {blend_count:,} rows")

# Generate meta-features on blend_val using LOADED models
bl = blend_val.withColumn("_idx", F.monotonically_increasing_id())
meta_cols = []

for mname, model in models.items():
    col = f"meta_{mname.lower()}"
    meta_cols.append(col)
    bl_pred = model.transform(bl).select("_idx", extract_p1("probability").alias(col))
    bl = bl.join(bl_pred, "_idx")

meta_asm = VectorAssembler(inputCols=meta_cols, outputCol="meta_features")
meta_train = meta_asm.transform(bl.select("_idx", "label", *meta_cols))

# Train lightweight meta-learner
print(f"Training meta-learner on {len(meta_cols)} base model predictions...")
meta_model = LogisticRegression(
    featuresCol="meta_features", labelCol="label",
    maxIter=100, regParam=0.01, family="binomial",
).fit(meta_train)

mc = meta_model.coefficients.toArray()
print(f"Meta-learner weights:")
for name, weight in zip(meta_cols, mc):
    print(f"  {name}: {weight:.4f}")
print(f"  intercept: {meta_model.intercept:.4f}")

# Test set predictions through stacking
ts = test_w.withColumn("_idx", F.monotonically_increasing_id())
for mname, model in models.items():
    col = f"meta_{mname.lower()}"
    ts_pred = model.transform(ts).select("_idx", extract_p1("probability").alias(col))
    ts = ts.join(ts_pred, "_idx")

meta_test = meta_asm.transform(ts.select("_idx", "label", *meta_cols))
stack_preds = meta_model.transform(meta_test)

evaluate_model(stack_preds, "E_C_Stacking_LR", time.time()-start,
    params={"base_models": "+".join(models.keys()), "meta_learner": "LR"},
    tags={"family": "ensemble", "type": "stacking"})

blend_val.unpersist()

ENSEMBLE C: STACKING (BLENDING) — LR META-LEARNER


Blend validation set: 1,844,517 rows
Training meta-learner on 4 base model predictions...


26/03/01 20:54:42 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/03/01 20:55:57 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/03/01 20:57:02 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:07 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:10 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:11 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:11 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:11 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:11 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:12 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:12 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
26/03/01 20:57:12 WARN DAGScheduler: Broadcasting larg

Meta-learner weights:
  meta_lr: 2.2068
  meta_dt: 2.3099
  meta_rf: 2.5917
  meta_gbt: 2.3189
  intercept: -5.3089


26/03/01 21:03:31 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/03/01 21:04:10 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 21:04:24 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/03/01 21:05:08 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 21:05:26 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/03/01 21:06:08 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 21:06:21 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/03/01 21:07:20 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB
26/03/01 21:07:31 WARN DAGScheduler: Broadcasting large task binary with size 6.3 MiB
26/03/01 21:08:20 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB



  E_C_Stacking_LR
  PR-AUC: 0.9988  |  ROC-AUC: 0.9996
  Denial P/R/F1: 0.9925 / 0.9878 / 0.9901
  Confusion:  TN=1,133,816  FP=2,961  FN=4,860  TP=392,203
  Time: 562.1s


26/03/01 21:08:22 WARN DAGScheduler: Broadcasting large task binary with size 7.0 MiB


DataFrame[label: double, features: vector, weight: double]

## Section 4 — Comprehensive Head-to-Head Comparison

All individual models and all ensemble variants compared side-by-side.

In [7]:
# ============================================================
# Cell 21: Final Leaderboard — All Models + All Ensembles
# ============================================================

print("=" * 100)
print("FINAL LEADERBOARD: ALL MODELS + ENSEMBLES (sorted by PR-AUC)")
print("=" * 100)

rows = []
for name, m in ALL_RESULTS.items():
    rows.append({
        "Model": name,
        "ROC-AUC": m.get("ROC-AUC", 0),
        "PR-AUC":  m.get("PR-AUC", 0),
        "Denial_F1": m.get("Denial_F1", 0),
        "D_Prec": m.get("Denial_Precision", 0),
        "D_Rec":  m.get("Denial_Recall", 0),
        "Accuracy": m.get("Accuracy", 0),
        "Time_s": m.get("Train_Time_s", 0),
    })

final_lb = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
final_lb.index = range(1, len(final_lb)+1)
final_lb.index.name = "Rank"

print(final_lb.to_string(float_format="%.4f"))

# Best per category
print("\n\nBest per category:")
best_individual = final_lb[~final_lb["Model"].str.startswith("E_")].iloc[0]
best_ensemble   = final_lb[final_lb["Model"].str.startswith("E_")]
print(f"  Best individual: {best_individual['Model']} (PR-AUC={best_individual['PR-AUC']:.4f})")
if len(best_ensemble) > 0:
    be = best_ensemble.iloc[0]
    print(f"  Best ensemble:   {be['Model']} (PR-AUC={be['PR-AUC']:.4f})")
    lift = be['PR-AUC'] - best_individual['PR-AUC']
    print(f"  Ensemble lift:   {lift:+.4f} ({lift/best_individual['PR-AUC']*100:+.2f}%)")

# Per-metric winners
print("\nBest model per metric:")
for col in ["ROC-AUC", "PR-AUC", "Denial_F1", "D_Prec", "D_Rec", "Accuracy"]:
    idx = final_lb[col].idxmax()
    print(f"  {col:<12} -> {final_lb.loc[idx, 'Model']:<30} ({final_lb.loc[idx, col]:.4f})")

FINAL LEADERBOARD: ALL MODELS + ENSEMBLES (sorted by PR-AUC)
                      Model  ROC-AUC  PR-AUC  Denial_F1  D_Prec  D_Rec  Accuracy    Time_s
Rank                                                                                      
1     E_B_WeightedVote_Top3   0.9997  0.9990     0.0000  0.0000 0.0000    0.7411    0.0181
2                    M6_GBT   0.9997  0.9989     0.9886  0.9869 0.9903    0.9941 3928.1332
3     M2_LogisticRegression   0.9997  0.9989     0.9880  0.9876 0.9884    0.9938  499.5773
4          E_A_SoftVote_All   0.9996  0.9988     0.9891  0.9887 0.9895    0.9943    0.2666
5           E_C_Stacking_LR   0.9996  0.9988     0.9901  0.9925 0.9878    0.9949  562.0941
6           M4_DecisionTree   0.9994  0.9982     0.9881  0.9860 0.9902    0.9938  439.3494
7           M5_RandomForest   0.9995  0.9981     0.9888  0.9898 0.9878    0.9942  871.3715
8              M3_LinearSVM   0.9994  0.9976     0.9890  0.9931 0.9850    0.9943  308.7870
9             M1_NaiveBayes  

In [8]:
# ============================================================
# Cell 22: Comprehensive Visualizations
# ============================================================

# Exclude baseline for cleaner plots
plot_df = final_lb[final_lb["Model"] != "M0_Baseline"].copy()
model_names = plot_df["Model"].tolist()
x = np.arange(len(model_names))

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# ── Panel A: PR-AUC + ROC-AUC bars ──
ax1 = fig.add_subplot(gs[0, 0])
w = 0.35
ax1.bar(x - w/2, plot_df["PR-AUC"], w, label="PR-AUC", color="#FF5722", alpha=0.85)
ax1.bar(x + w/2, plot_df["ROC-AUC"], w, label="ROC-AUC", color="#2196F3", alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(model_names, rotation=45, ha="right", fontsize=7)
ax1.set_ylabel("Score")
ax1.set_title("PR-AUC vs ROC-AUC", fontweight="bold")
ax1.legend()
ax1.grid(axis="y", alpha=0.3)

# ── Panel B: Denial Precision vs Recall scatter ──
ax2 = fig.add_subplot(gs[0, 1])
colors_map = plt.cm.tab10(np.linspace(0, 1, len(plot_df)))
for i, (_, row) in enumerate(plot_df.iterrows()):
    ax2.scatter(row["D_Rec"], row["D_Prec"], s=120, c=[colors_map[i]], 
               edgecolors="black", zorder=5)
    ax2.annotate(row["Model"].split("_",1)[-1], (row["D_Rec"], row["D_Prec"]),
                textcoords="offset points", xytext=(6, 4), fontsize=6)
for f1 in [0.3, 0.5, 0.7]:
    rv = np.linspace(0.01, 1, 100)
    pv = f1 * rv / (2 * rv - f1)
    valid = (pv > 0) & (pv <= 1)
    ax2.plot(rv[valid], pv[valid], "--", color="gray", alpha=0.3, linewidth=0.7)
ax2.set_xlabel("Denial Recall")
ax2.set_ylabel("Denial Precision")
ax2.set_title("Precision-Recall Trade-off", fontweight="bold")
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
ax2.grid(alpha=0.3)

# ── Panel C: Training time ──
ax3 = fig.add_subplot(gs[1, 0])
times = plot_df["Time_s"].values
colors_t = ["#4CAF50" if t < 60 else "#FF9800" if t < 300 else "#F44336" for t in times]
ax3.barh(model_names, times, color=colors_t, alpha=0.8)
ax3.set_xlabel("Training Time (seconds)")
ax3.set_title("Training Time Comparison", fontweight="bold")
ax3.grid(axis="x", alpha=0.3)

# ── Panel D: Confusion matrix for top 3 models ──
ax4 = fig.add_subplot(gs[1, 1])
top3_models = plot_df.head(3)["Model"].tolist()
bar_data = []
for mn in top3_models:
    cm = ALL_RESULTS[mn]["Confusion"]
    total_cm = cm["TP"]+cm["FP"]+cm["FN"]+cm["TN"]
    bar_data.append({
        "Model": mn.split("_",1)[-1],
        "TP%": cm["TP"]/total_cm*100,
        "FP%": cm["FP"]/total_cm*100,
        "FN%": cm["FN"]/total_cm*100,
        "TN%": cm["TN"]/total_cm*100,
    })
bar_df = pd.DataFrame(bar_data)
x4 = np.arange(len(bar_df))
w4 = 0.2
ax4.bar(x4-1.5*w4, bar_df["TP%"], w4, label="TP", color="#4CAF50")
ax4.bar(x4-0.5*w4, bar_df["TN%"], w4, label="TN", color="#2196F3")
ax4.bar(x4+0.5*w4, bar_df["FP%"], w4, label="FP", color="#FF9800")
ax4.bar(x4+1.5*w4, bar_df["FN%"], w4, label="FN", color="#F44336")
ax4.set_xticks(x4)
ax4.set_xticklabels(bar_df["Model"], fontsize=8)
ax4.set_ylabel("% of test set")
ax4.set_title("Top 3: Confusion Breakdown", fontweight="bold")
ax4.legend(fontsize=8)
ax4.grid(axis="y", alpha=0.3)

plt.savefig(f"{DATA_DIR}/comprehensive_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: comprehensive_comparison.png")

Saved: comprehensive_comparison.png


In [9]:
# ============================================================
# Cell 23: Feature Importance — Cross-Model Comparison
# ============================================================

feature_names = feature_meta.get("assembler_inputs", [])
if len(feature_names) < feature_dim:
    feature_names = [f"feature_{i}" for i in range(feature_dim)]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, (name, title, color) in zip(axes, [
    ("DT", "Decision Tree", "#2196F3"),
    ("RF", "Random Forest", "#4CAF50"),
    ("GBT", "GBT", "#F44336"),
]):
    if name in models:
        imp = models[name].featureImportances.toArray()
        top_idx = np.argsort(imp)[::-1][:12]
        names = [feature_names[i] if i < len(feature_names) else f"f_{i}" for i in top_idx]
        ax.barh(range(12), imp[top_idx][::-1], color=color, alpha=0.8)
        ax.set_yticks(range(12))
        ax.set_yticklabels(names[::-1], fontsize=7)
        ax.set_title(title, fontweight="bold")
    else:
        ax.set_title(f"{title} (not loaded)")

plt.suptitle("Feature Importance — Cross-Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

# Agreement analysis
sets = {}
for name in ["DT", "RF", "GBT"]:
    if name in models:
        imp = models[name].featureImportances.toArray()
        sets[name] = set(np.argsort(imp)[::-1][:10])

print("\nTop-10 feature overlap:")
names_list = list(sets.keys())
for i, m1 in enumerate(names_list):
    for m2 in names_list[i+1:]:
        overlap = len(sets[m1] & sets[m2])
        print(f"  {m1} vs {m2}: {overlap}/10 shared")


Top-10 feature overlap:
  DT vs RF: 3/10 shared
  DT vs GBT: 7/10 shared
  RF vs GBT: 3/10 shared


## Section 5 — Threshold Optimization

### Why 0.5 is wrong for imbalanced data

The default threshold (P>0.5 = positive) assumes balanced classes and equal misclassification costs. For HMDA:
- Missing a discriminatory denial (FN) is worse than a false alarm (FP)
- Classes are 80/20, not 50/50

We sweep thresholds to find: (a) max Denial F1, (b) 80% recall operating point.

In [10]:
# ============================================================
# Cell 24: Threshold Optimization for Best Model
# ============================================================

print("=" * 70)
print("THRESHOLD OPTIMIZATION")
print("=" * 70)

final_lb = pd.DataFrame([
    {"Model": name, "ROC-AUC": m.get("ROC-AUC",0), "PR-AUC": m.get("PR-AUC",0),
     "Denial_F1": m.get("Denial_F1",0), "D_Prec": m.get("Denial_Precision",0),
     "D_Rec": m.get("Denial_Recall",0), "Accuracy": m.get("Accuracy",0),
     "Time_s": m.get("Train_Time_s",0)}
    for name, m in ALL_RESULTS.items()
]).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
final_lb.index = range(1, len(final_lb)+1)
final_lb.index.name = "Rank"

best_overall = final_lb.iloc[0]["Model"]
print(f"Best model: {best_overall}")

# Get predictions from best available model
if "GBT" in models:
    thresh_model = models["GBT"]
elif "RF" in models:
    thresh_model = models["RF"]
else:
    thresh_model = list(models.values())[0]

thresh_preds = thresh_model.transform(test_w)

pdf = (thresh_preds
    .select("label", extract_p1("probability").alias("prob"))
    .sample(fraction=0.1, seed=42).toPandas())

y_true = pdf["label"].values
y_prob = pdf["prob"].values

thresholds = np.arange(0.10, 0.91, 0.02)
results = []
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    prec = tp/(tp+fp) if (tp+fp) > 0 else 0
    rec  = tp/(tp+fn) if (tp+fn) > 0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
    results.append({"threshold": t, "precision": prec, "recall": rec, "f1": f1})

res_df = pd.DataFrame(results)
best_f1_idx = res_df["f1"].idxmax()
optimal_t = res_df.loc[best_f1_idx, "threshold"]

print(f"\nOptimal threshold (max Denial F1): {optimal_t:.2f}")
print(f"  Precision: {res_df.loc[best_f1_idx, 'precision']:.4f}")
print(f"  Recall:    {res_df.loc[best_f1_idx, 'recall']:.4f}")
print(f"  F1:        {res_df.loc[best_f1_idx, 'f1']:.4f}")

for _, row in res_df.iterrows():
    if row["recall"] >= 0.80:
        print(f"\nThreshold for 80% recall: {row['threshold']:.2f}")
        print(f"  Precision: {row['precision']:.4f}, F1: {row['f1']:.4f}")
        break

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(res_df["threshold"], res_df["precision"], "b-", lw=2, label="Precision")
ax.plot(res_df["threshold"], res_df["recall"], "r-", lw=2, label="Recall")
ax.plot(res_df["threshold"], res_df["f1"], "g-", lw=2, label="Denial F1")
ax.axvline(x=0.5, color="gray", ls="--", alpha=0.5, label="Default (0.5)")
ax.axvline(x=optimal_t, color="green", ls="--", alpha=0.7, label=f"Optimal ({optimal_t:.2f})")
ax.set_xlabel("Classification Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Optimization — {best_overall}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/threshold_optimization.png", dpi=150, bbox_inches="tight")
plt.show()

THRESHOLD OPTIMIZATION
Best model: E_B_WeightedVote_Top3



Optimal threshold (max Denial F1): 0.62
  Precision: 0.9925
  Recall:    0.9885
  F1:        0.9905

Threshold for 80% recall: 0.10
  Precision: 0.9210, F1: 0.9588


In [11]:
# ============================================================
# Cell 25: Error Analysis & Model Agreement
# ============================================================

print("=" * 70)
print("ERROR ANALYSIS: WHERE DO MODELS DISAGREE?")
print("=" * 70)

# Using our pandas-based ALL_PREDICTIONS from diversity analysis
# (Already have base_pdf and pred_cols from Cell 25)

y = base_pdf["label"].values

print("\nModel error rates (on sampled data):")
for col in pred_cols:
    mname = col.replace("pred_", "")
    preds = (base_pdf[col].values > 0.5).astype(int)
    err = (preds != y).mean()
    print(f"  {mname:<15} error rate: {err:.4f} ({err*100:.1f}%)")

# Find "hard cases" — samples ALL models get wrong
all_wrong = np.ones(len(y), dtype=bool)
for col in pred_cols:
    preds = (base_pdf[col].values > 0.5).astype(int)
    all_wrong &= (preds != y)

n_all_wrong = all_wrong.sum()
print(f"\nHard cases (ALL models wrong): {n_all_wrong} ({n_all_wrong/len(y)*100:.1f}%)")
print("  These represent genuinely ambiguous loan decisions.")
print("  No ensemble can fix these — they need better features or domain rules.")

# GBT vs LR comparison (if both available)
if "pred_GBT" in base_pdf.columns and "pred_LR" in base_pdf.columns:
    lr_preds_arr = (base_pdf["pred_LR"].values > 0.5).astype(int)
    gbt_preds_arr = (base_pdf["pred_GBT"].values > 0.5).astype(int)

    gbt_right_lr_wrong = ((gbt_preds_arr == y) & (lr_preds_arr != y)).sum()
    lr_right_gbt_wrong = ((lr_preds_arr == y) & (gbt_preds_arr != y)).sum()

    print(f"\nGBT vs LR:")
    print(f"  GBT right, LR wrong: {gbt_right_lr_wrong} (non-linearity helped)")
    print(f"  LR right, GBT wrong: {lr_right_gbt_wrong} (GBT overfit)")
    print(f"  Net GBT advantage:   {gbt_right_lr_wrong - lr_right_gbt_wrong:+d}")

ERROR ANALYSIS: WHERE DO MODELS DISAGREE?

Model error rates (on sampled data):
  NaiveBayes      error rate: 0.0079 (0.8%)
  LR              error rate: 0.0063 (0.6%)
  SVM             error rate: 0.9942 (99.4%)
  DT              error rate: 0.0059 (0.6%)
  RF              error rate: 0.0059 (0.6%)
  GBT             error rate: 0.0057 (0.6%)
  MLP             error rate: 0.0065 (0.7%)

Hard cases (ALL models wrong): 0 (0.0%)
  These represent genuinely ambiguous loan decisions.
  No ensemble can fix these — they need better features or domain rules.

GBT vs LR:
  GBT right, LR wrong: 164 (non-linearity helped)
  LR right, GBT wrong: 121 (GBT overfit)
  Net GBT advantage:   +43


## Section 6 — MLflow Hyperparameter Analysis

### How to use MLflow for hyperparameter tuning (interview-ready)

MLflow tracks experiments via:
1. **Runs**: Each training attempt (model + hyperparameters + metrics)
2. **Parameters**: What was configured (`regParam=0.01`, `maxDepth=8`)
3. **Metrics**: What was measured (`PR-AUC=0.72`, `training_time=45s`)
4. **Artifacts**: What was produced (model files, plots, confusion matrices)

**Key MLflow commands**:
```python
mlflow.set_experiment("my_experiment")
with mlflow.start_run(run_name="GBT_v1"):
    mlflow.log_param("maxDepth", 6)
    mlflow.log_metric("PR_AUC", 0.72)
    mlflow.spark.log_model(model, "model")
```

**To view**: `mlflow ui --port 5000` → opens browser dashboard at localhost:5000

### What MLflow shows us about our experiments

All the hyperparameter combinations we tried are logged. We can analyze:
- Which hyperparameters matter most (parameter importance)
- Whether there are interactions (e.g., high depth + low learning rate = best)
- Diminishing returns (more trees past some point doesn't help)

In [12]:
# ============================================================
# Cell 26: MLflow Experiment Summary & Hyperparameter Analysis
# ============================================================

print("=" * 70)
print("MLFLOW EXPERIMENT SUMMARY")
print("=" * 70)

if MLFLOW_AVAILABLE:
    # Query all runs from our experiment
    experiment = mlflow.get_experiment_by_name("HMDA_2023_Model_Training")

    if experiment:
        runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
        print(f"\nTotal runs logged: {len(runs)}")

        if len(runs) > 0:
            # Show key columns
            display_cols = ["tags.model_name", "metrics.PR_AUC", "metrics.ROC_AUC",
                          "metrics.Denial_F1", "metrics.Train_Time_s"]
            available_cols = [c for c in display_cols if c in runs.columns]

            if available_cols:
                runs_display = runs[available_cols].sort_values(
                    "metrics.PR_AUC", ascending=False).head(15)
                print("\nTop 15 runs by PR-AUC:")
                print(runs_display.to_string(index=False))

            print(f"\nMLflow UI: Run 'mlflow ui --port 5000' to explore interactively")
            print(f"  Tracking dir: {mlflow.get_tracking_uri()}")
    else:
        print("No experiment found.")
else:
    print("MLflow not available. All results stored in ALL_RESULTS dict.")
    print("\nTo enable MLflow:")
    print("  pip install mlflow")
    print("  Then re-run this notebook.")

# Always print from our in-memory storage
print("\n" + "=" * 70)
print("IN-MEMORY RESULTS SUMMARY")
print("=" * 70)
for name in sorted(ALL_RESULTS.keys()):
    m = ALL_RESULTS[name]
    print(f"  {name:<30} PR-AUC={m.get('PR-AUC',0):.4f}  "
          f"Denial_F1={m.get('Denial_F1',0):.4f}  "
          f"Time={m.get('Train_Time_s',0):.0f}s")

MLFLOW EXPERIMENT SUMMARY
MLflow not available. All results stored in ALL_RESULTS dict.

To enable MLflow:
  pip install mlflow
  Then re-run this notebook.

IN-MEMORY RESULTS SUMMARY
  E_A_SoftVote_All               PR-AUC=0.9988  Denial_F1=0.9891  Time=0s
  E_B_WeightedVote_Top3          PR-AUC=0.9990  Denial_F1=0.0000  Time=0s
  E_C_Stacking_LR                PR-AUC=0.9988  Denial_F1=0.9901  Time=562s
  M0_Baseline                    PR-AUC=0.2589  Denial_F1=0.0000  Time=0s
  M1_NaiveBayes                  PR-AUC=0.9937  Denial_F1=0.9846  Time=61s
  M2_LogisticRegression          PR-AUC=0.9989  Denial_F1=0.9880  Time=500s
  M3_LinearSVM                   PR-AUC=0.9976  Denial_F1=0.9890  Time=309s
  M4_DecisionTree                PR-AUC=0.9982  Denial_F1=0.9881  Time=439s
  M5_RandomForest                PR-AUC=0.9981  Denial_F1=0.9888  Time=871s
  M6_GBT                         PR-AUC=0.9989  Denial_F1=0.9886  Time=3928s
  M7_MLP                         PR-AUC=0.9846  Denial_F1=0.98

In [13]:
# ============================================================
# Cell 27: Save All Final Artifacts
# ============================================================

print("=" * 70)
print("SAVING ALL ARTIFACTS")
print("=" * 70)

def sanitize(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, dict): return {k: sanitize(v) for k, v in obj.items()}
    return obj

# Save all metrics (individual + ensemble)
results_path = f"{DATA_DIR}/model_results.json"
with open(results_path, "w") as f:
    json.dump(sanitize(ALL_RESULTS), f, indent=2, default=str)
print(f"  Saved: {results_path}")

# Save leaderboard
lb_path = f"{DATA_DIR}/model_leaderboard.csv"
final_lb.to_csv(lb_path, index=True)
print(f"  Saved: {lb_path}")

# Save optimal threshold
thresh_path = f"{DATA_DIR}/optimal_threshold.json"
with open(thresh_path, "w") as f:
    json.dump({
        "best_model": best_overall,
        "optimal_threshold_f1": float(optimal_t),
        "optimal_precision": float(res_df.loc[best_f1_idx, "precision"]),
        "optimal_recall": float(res_df.loc[best_f1_idx, "recall"]),
        "optimal_f1": float(res_df.loc[best_f1_idx, "f1"]),
    }, f, indent=2)
print(f"  Saved: {thresh_path}")

print(f"\nAll artifacts saved to {DATA_DIR}/")

SAVING ALL ARTIFACTS
  Saved: /Users/adi/Desktop/assignmt/project/data/processed/model_results.json
  Saved: /Users/adi/Desktop/assignmt/project/data/processed/model_leaderboard.csv
  Saved: /Users/adi/Desktop/assignmt/project/data/processed/optimal_threshold.json

All artifacts saved to /Users/adi/Desktop/assignmt/project/data/processed/


## Section 7 — Why Models Performed As They Did (Dataset-Specific Analysis)

### Expected ranking (from ML theory + HMDA properties):

```
GBT >= RF > DT > LR >= SVM > MLP > NB > Baseline
Stacking >= Weighted Voting >= Soft Voting >= Best Individual
```

### Model-by-model explanation

**Naive Bayes (weakest real model)**:
The independence assumption is badly violated in HMDA — income, loan amount, property value, and LTV are all correlated. NB can't model this, so it gets a rough approximation. However, it still beats the baseline because even under independence, high income + low DTI + conventional loan type all individually point toward approval.

**Logistic Regression (strong linear baseline)**:
Our NB3 feature engineering specifically helped LR: log transforms linearized skewed financial relationships, loan_to_income_ratio captured a key interaction, and StandardScaler (withStd=True) put all features on comparable scales. LR's performance reflects *our feature engineering quality* as much as the algorithm.

**Linear SVM (similar to LR)**:
With our well-scaled features, the maximum-margin objective produces a similar boundary to LR's maximum-likelihood objective. The small differences come from the hinge loss vs log-loss distinction — hinge loss is more sensitive to points near the boundary.

**Decision Tree (captures thresholds, but overfits)**:
DT shines on individual lending rules (DTI > 43%, LTV > 80%) but its high variance means slight training data changes produce different trees. On 6M rows, the tree structure is more stable than on small data, but it still memorizes noise at deep levels.

**Random Forest (stable, robust)**:
Averaging 100+ trees reduces the DT's variance dramatically. On HMDA, RF captures the same threshold effects as DT but without the noise. The random feature subsets force trees to explore different splitting strategies — some trees might focus on financial ratios, others on loan type and geography.

**GBT (likely winner among individuals)**:
GBT's sequential error correction is perfectly suited to HMDA's challenge: easy cases (very high/low income) are classified early, and later trees focus on the **marginal cases** where denial vs approval depends on subtle feature combinations. These edge cases are exactly where fair lending issues hide.

**MLP (competitive but less suited)**:
MLP can learn feature interactions via hidden layers, but it lacks the inductive biases that help trees on tabular data (automatic threshold detection, handling of mixed types, robustness to feature scale). MLP also can't use class weights in PySpark, hurting its denial recall.

### General rules: which models for which scenarios

| Scenario | Best model family | Why |
|:---------|:-----------------|:----|
| Tabular data, mixed types | GBT / XGBoost / LightGBM | Built for this; handles everything |
| Need interpretability | LR or shallow DT | Coefficients / rules are human-readable |
| Very small data (<1K) | LR with regularization | Fewer parameters → less overfitting |
| Very large data (>100M) | LightGBM or RF | Histogram binning / embarrassingly parallel |
| Text / NLP | Naive Bayes → BERT | Sparse features → dense representations |
| Images | CNN (not applicable here) | Spatial structure needs convolutions |
| Need fast inference | LR (single matrix multiply) | O(d) prediction vs O(trees * depth) |
| Need maximum accuracy | Stacking ensemble of GBT + RF + LR | Diverse errors cancel |
| Regulatory/fair lending | LR + SHAP on GBT | Interpret + best performance |

### Interview answers this notebook provides

1. *"Walk through your model selection process"* → Simple-to-complex ladder with justification
2. *"Why does GBT beat RF?"* → Sequential correction targets hard cases; RF is limited by independent trees
3. *"Explain AdaBoost vs Gradient Boosting vs XGBoost"* → Sample reweighting → residual fitting → regularized 2nd-order optimization
4. *"How do you build an ensemble?"* → Diversity analysis → select complementary models → choose architecture → validate lift
5. *"How do you handle class imbalance?"* → Class weights (not SMOTE) because abundant minority samples
6. *"What metric for imbalanced data?"* → PR-AUC (not ROC-AUC) because it focuses on the positive class
7. *"How does MLflow help?"* → Tracks all experiments for reproducibility, comparison, and hyperparameter analysis
8. *"When would you use LR over GBT?"* → When interpretability is required, data is very small, or fast inference is needed
9. *"Why might ensembles NOT help?"* → When models make correlated errors (same hard cases fail for everyone)
10. *"Explain the bias-variance tradeoff"* → DT=high variance, LR=high bias, RF reduces variance, GBT reduces bias

---

### NEXT: Notebook 5 — Fair Lending Audit & Final Evaluation

- Evaluate model fairness across protected demographic groups (race, sex, ethnicity)
- Disparate impact analysis
- SHAP / feature contribution analysis
- Final production-ready model export

In [14]:
# ============================================================
# Cleanup
# ============================================================
train_w.unpersist()
test_w.unpersist()
spark.stop()
print("Notebook 4b complete. Spark stopped.")

Notebook 4b complete. Spark stopped.
